# 第 43 章：Capstone 试教材料与课程运行包

这个 notebook 用一个极小的 trial-teaching feedback 表，对比“只按 revision load 排序”的 naive baseline 和“先检查 privacy / runtime / rubric / materials gate”的课程运行 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_trial_teaching_feedback_demo.csv')


def load_trials(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['revision_load_score'] = float(row['revision_load_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_trials(DATA_PATH)
print(f'Loaded {len(rows)} trial-teaching cards from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Student readiness:', ordered_counts(rows, 'student_readiness_status'))
print('Notebook runtime:', ordered_counts(rows, 'notebook_runtime_status'))
print('Rubric calibration:', ordered_counts(rows, 'rubric_calibration_status'))
print('Privacy boundary:', ordered_counts(rows, 'privacy_boundary_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['trial_id']
    for row in rows
    if row['reference_route'] == 'ready_for_rollout'
)

top3_low_revision = sorted(rows, key=lambda row: row['revision_load_score'])[:3]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_for_rollout'
    for row in top3_low_revision
)
baseline_blocked_hits = sum(
    row['reference_route'] == 'blocked_before_trial'
    for row in top3_low_revision
)
baseline_ready_precision = baseline_ready_hits / len(top3_low_revision)
baseline_blocked_intrusion = baseline_blocked_hits / len(top3_low_revision)

print('Reference ready-for-rollout trials:', reference_ready_ids)
print('Top-3 by low revision load:')
for row in top3_low_revision:
    print(
        f"  {row['trial_id']}: load={row['revision_load_score']:.2f}, "
        f"route={row['reference_route']}, theme={row['session_theme']}"
    )
print('Baseline rollout precision:', round(baseline_ready_precision, 3))
print('Baseline blocked intrusion:', round(baseline_blocked_intrusion, 3))


In [ ]:
def route_trial(row):
    if row['privacy_boundary_status'] != 'clear':
        return 'blocked_before_trial', 'privacy, sharing, or external-AI boundary is not clear'
    if row['notebook_runtime_status'] == 'fails':
        return 'blocked_before_trial', 'notebook does not run in the course environment'
    if row['rubric_calibration_status'] != 'calibrated':
        return 'calibrate_rubric', 'rubric or TA norming still needs calibration'
    if row['handout_clarity_status'] != 'clear':
        return 'revise_materials', 'student handout is not yet clear enough'
    if row['student_readiness_status'] == 'low':
        return 'revise_materials', 'students need more scaffolding before rollout'
    if row['ta_feedback_status'] != 'ready':
        return 'revise_materials', 'TA feedback format or examples are not ready'
    if row['revision_load_score'] > 0.45:
        return 'revise_materials', 'revision load is too high for immediate rollout'
    return 'ready_for_rollout', 'materials, runtime, rubric, feedback, and boundaries are ready'


routed_rows = []
for row in rows:
    route, reason = route_trial(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Trial workflow routes:')
for row in routed_rows:
    print(
        f"  {row['trial_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_for_rollout']
revise_queue = [row for row in routed_rows if row['route'] == 'revise_materials']
calibration_queue = [row for row in routed_rows if row['route'] == 'calibrate_rubric']
blocked_queue = [row for row in routed_rows if row['route'] == 'blocked_before_trial']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'ready_for_rollout'
    for row in ready_queue
) / max(len(ready_queue), 1)
ready_recall = sum(
    row['route'] == 'ready_for_rollout' and row['reference_route'] == 'ready_for_rollout'
    for row in routed_rows
) / max(len(reference_ready_ids), 1)

print('Ready-for-rollout queue:', [row['trial_id'] for row in ready_queue])
print('Revise-materials queue:', [row['trial_id'] for row in revise_queue])
print('Calibrate-rubric queue:', [row['trial_id'] for row in calibration_queue])
print('Blocked-before-trial queue:', [row['trial_id'] for row in blocked_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured rollout precision:', round(ready_precision, 3))
print('Structured rollout recall:', round(ready_recall, 3))


In [ ]:
def trial_revision_steps(row):
    steps = []
    if row['privacy_boundary_status'] != 'clear':
        steps.append('先明确数据、学生材料和外部 AI 服务的共享边界，暂时不要试教。')
    if row['notebook_runtime_status'] == 'fails':
        steps.append('在课程环境里从头修复 notebook，记录依赖、路径和运行时间。')
    elif row['notebook_runtime_status'] == 'slow':
        steps.append('压缩数据或缓存中间结果，让 notebook 能在课堂窗口内完成。')
    if row['rubric_calibration_status'] != 'calibrated':
        steps.append('组织一次 TA norming session，用 2 到 3 个样例校准评分尺度。')
    if row['handout_clarity_status'] != 'clear':
        steps.append('重写 handout 的第一步、交付物、截止时间和评分前 gate。')
    if row['student_readiness_status'] == 'low':
        steps.append('增加前置 mini-lab 或 checkpoint，避免学生直接跳进期末项目。')
    if row['ta_feedback_status'] != 'ready':
        steps.append('补 TA feedback 模板、示范评语和无法判断时的升级路径。')
    if row['revision_load_score'] > 0.45:
        steps.append('把材料修订拆成负责人明确的一周任务，不要一次性重写所有环节。')
    return steps or ['可以进入正式 rollout；仍需保留运行日志和学生反馈表。']


for row in routed_rows:
    if row['route'] != 'ready_for_rollout':
        print(f"\n{row['trial_id']} -> {row['route']}")
        for step in trial_revision_steps(row):
            print(' -', step)


In [ ]:
trial_assistant_prompt = '''你是我的 capstone 试教助手。

任务：
1. 先阅读 trial-teaching feedback table，不要只看 revision load；
2. 先检查 privacy boundary 和 notebook runtime；
3. 再检查 rubric calibration、handout clarity、student readiness 和 TA feedback；
4. 把每个环节 route 到 ready_for_rollout、revise_materials、
   calibrate_rubric 或 blocked_before_trial；
5. 对每个非 ready 环节输出下一轮课程修订 checklist。

输出格式：
- Ready-for-rollout sessions
- Revise-materials queue
- Calibrate-rubric queue
- Blocked-before-trial queue
- Revision checklist by session
'''
print(trial_assistant_prompt)


In [ ]:
student_handout_sections = [
    'Project goal and success criterion',
    'Timeline and checkpoint dates',
    'Data and privacy boundary',
    'Notebook runtime and reproducibility requirements',
    'Rubric, gates, and minimum evidence lines',
    'AI-use policy and disclosure examples',
    'Help path, TA office hours, and human signoff',
    'Final submission checklist',
]

trial_snapshot = {
    'dataset': DATA_PATH.name,
    'n_trial_cards': len(rows),
    'baseline_top3_rollout_precision': round(baseline_ready_precision, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_for_rollout': [row['trial_id'] for row in ready_queue],
    'revise_materials': [row['trial_id'] for row in revise_queue],
    'calibrate_rubric': [row['trial_id'] for row in calibration_queue],
    'blocked_before_trial': [row['trial_id'] for row in blocked_queue],
    'python_version': platform.python_version(),
}

print('Student handout sections:')
for section in student_handout_sections:
    print(' -', section)

print('\nTrial snapshot:')
for key, value in trial_snapshot.items():
    print(f'  {key}: {value}')
